#Practica 6:
Aprovechando el dataframe de semestre (enero-junio 2020), cargar las tablas del pdf "Tablas Telefonia.pdf", hacer merge con las 3 tablas que contiene el pdf, para obtener la dimension franja, la dimensión Descripción, promociones a aplicar, aplicar el descuento al importe factura según su zona.
- Primero aplicar el descuento por Zona (lo tenemos en la tabla "Promociones a aplicar")
- Guardar en un csv, la suma de importe neto por Franja.
- Guardar en un csv, el numero de facturas por descripción de la satisfacción.

In [ ]:
#Solución practica 6
#0. Importar las librerias
import pdfplumber
import pandas as pd

#1. Ajustar la ruta del archivo de excel
ruta_fichero = r"C:\PCAD\Datos Telefonia Separados Meses Comerciales URL.xlsx"

#2. Descargar los dataframes de los meses (enero-junio) 
fras_enero = pd.read_excel(ruta_fichero,sheet_name="Facturación Enero 2020", header=2)
fras_febrero = pd.read_excel(ruta_fichero,sheet_name="Facturación Febrero 2020", header=0)
fras_marzo = pd.read_excel(ruta_fichero,sheet_name="Facturación Marzo 2020", header=1)
fras_abril = pd.read_excel(ruta_fichero,sheet_name="Facturación Abril 2020", header=1)
fras_mayo = pd.read_excel(ruta_fichero,sheet_name="Facturación Mayo 2020", header=1)
fras_junio = pd.read_excel(ruta_fichero,sheet_name="Facturación Junio 2020", header=2)

#3. Concatenar los 6 meses
fras_semestre = pd.concat([fras_enero,fras_febrero,fras_marzo,fras_abril, fras_mayo, fras_junio],ignore_index=True)

#4. Comprobar si el dataframe del semestre esta ok
fras_semestre

#5. Eliminar las filas en blanco
fras_semestre = fras_semestre.dropna(how='all')

#6. Cargar el dataframe de localidades, necesario para el calculo del dto %
localidades = pd.read_excel(ruta_fichero,sheet_name="Localidad", header=0)

#7. Dividir la columna Localidad por un caracter (#), quedarnos con la segunda columna = indice 1 + quitar los espacios en blanco
localidades['Localidad'] = localidades['Localidad'].str.split("#").str[1].str.strip()

#8. Dejar solo 2 columnas
localidades = localidades[['Localidad','Zona']]

#9. Comprobar el dataframe de localidades
localidades

#10. Merge df semestre con localidades
fras_semestre = fras_semestre.merge(localidades,on="Localidad",how='inner')

#11. Ajustar la ruta del fichero pdf
ruta_pdf = r"C:\PCAD\Tablas Telefonía.pdf"

#12. Crear una lista vacia para almacenar las tablas extraidas
tablas = []

#13. Crear un bucle para ir cargando las tablas, miraremos las diferentes tablas, de las diferentes paginas
with pdfplumber.open(ruta_pdf) as pdf:
    for page in pdf.pages:     #Es que me recorra todas las paginas del pdf
        #Extraer las tablas de la pagina que estoy recorriendo
        tablas_extraidas = page.extract_tables()
        #Vamos a recorrer la colección tablas_extraidas
        for tabla in tablas_extraidas:
            df = pd.DataFrame(tabla)
            tablas.append(df)

#14. Comprobar el numero de tablas cargadas
len(tablas)

#15. Acceder al dataframe resultante, despues de asignar a una referencia
franjas = tablas[0]

#16. Eliminar las columnas no validas
franjas = franjas.drop([1,2,4,5,7,8], axis=1)

#17. Eliminar la primera fila, es decir indice 0
franjas = franjas.drop([0], axis=0).reset_index(drop=True)

#18. Poner los nombres de las columnas
franjas = franjas.rename(columns={
    0:"Decada",
    3:"Franja",
    6:"Descripción"
})

#19. Ajustar el campo decada com numero entero
franjas['Decada'] = franjas['Decada'].astype("Int64")

#20. Ajustar el dataframe de descripciones
descripcion = tablas[1]

#21. Eliminar las 4 columnas sobrantes
descripcion = descripcion.drop([1,2,4,5], axis=1).reset_index(drop=True)

#22. Eliminar la primera fila
descripcion = descripcion.drop([0], axis= 0).reset_index(drop=True)

#23. Poner los nombres de las columas
descripcion = descripcion.rename(columns={
    0:"Satisfacción Cliente",
    3:"Descripción Satisfacción"
})

#24. Convertir Satisfaccion en enter
descripcion['Satisfacción Cliente'] = descripcion['Satisfacción Cliente'].astype("int64")

#25. Crear dataframe de promociones
promociones = tablas[2]

#26. Eliminar las columnas sobrantes
promociones = promociones.drop([1,2,4,5,7,8],axis=1).reset_index(drop=True)

#27. Eliminar la primera fila (indice = 0)
promociones = promociones.drop([0],axis=0).reset_index(drop=True)

#28. Renombrar las columnas
promociones = promociones.rename(columns={
    0:"Descripción promoción",
    3:"Aplicar a.",
    6:"% Dto."
})

#29. Ajustar el %Dto para poder calcular 4,50%
promociones['% Dto.'] = (
    promociones['% Dto.']
    .str.replace("%","")
    .str.replace(",",".")
    .astype(float) / 100
)

#30. Ver el tipo de datos del dataframe
#fras_semestre.info()

#31. Calcular la edad + decada en el dataframe de semestre
#31.1 Forzar al campo Fecha Nacimiento como datetime
fras_semestre['Fecha Nacimiento'] = pd.to_datetime(fras_semestre['Fecha Nacimiento'] )
#31.2 Calcular el campo Edad
fras_semestre['Edad'] = ((pd.to_datetime("today")-fras_semestre["Fecha Nacimiento"]).dt.days // 365).astype("Int64")
#31.3 Calcular la Decada + convertirla a integer
fras_semestre['Decada'] = ((fras_semestre['Edad'] // 10) * 10).astype("Int64")

#32. Hacer merge de fras_semestre con franjas, por defecto el how es 'inner'
fras_semestre = (fras_semestre
    .merge(franjas, on="Decada")
    .merge(descripcion,left_on="Satisfacción", right_on="Satisfacción Cliente")
    .merge(promociones, left_on="Zona", right_on="Aplicar a.")
)

#33. Calculo del importe neto aplicado el descuento por zona
fras_semestre['Importe factura neto'] = round(fras_semestre['Importe factura'] * (1-fras_semestre['% Dto.']),2)

#34. Quedarme con los 3 campos que necesito
resultado = fras_semestre[['Importe factura neto','Franja','Descripción Satisfacción']]

#35. Agrupar el importe factura neto por franjas
importe_franjas = resultado.groupby(['Franja'])['Importe factura neto'].sum().reset_index()

#36. Agrupar el numero de facturas por descripción de la satisfacción
facturas_descripcion = resultado.groupby(['Descripción Satisfacción'])['Importe factura neto'].count().reset_index(name='Numero fras.')

#37. Guardar en csv el dataframe importe_franjas
importe_franjas.to_csv(r"C:\PCAD\Importe Franjas.csv")

#38. Guardar en csv el dataframe facturas descripción
facturas_descripcion.to_csv(r"C:\PCAD\Fras Descripcion.csv")

print("Terminado con éxito")

Pendiente para terminar:
- Calcular la edad y la decada (aprovechar codigo)
- Hacer los 4 merges (franjas, Satisfacción, localidades, promociones)
- calcular el descuento (Opción A vectorizado, la opción B con Apply)
- hago las 2 agrupaciones (convertirlas a dataframe)
- Guardarlas en csv.

In [ ]:
resultado.groupby(['Descripción Satisfacción'])['Importe factura neto'].count().reset_index()

Formularios en Python con el modulo tkinter
- Es una interficie grafica de escritorio, es una ventana donde podremos colocar widgets.
- Widgets principales:
- Label = Etiqueta
- Entry = Caja de texto (el usuario puede escribir)
- Button = Botón para ejecutar un codigo
- Checkbutton / Radiobutton Casillas de verificación

Como se estructura un formulario
- Ventana Raiz (tk) = es el contenedor principal
- Gestor de geometria, se usa para posicionar elementos
- Bucle principal o mainloop

In [ ]:
#0. Importar las librerias tkinter es nativo
import tkinter as tk

#1. Crear la ventana principal
root = tk.Tk()
root.title("Dashboard de KPIs")

#2. Ajustar las medidas de la ventana en px, ajustar el left = 100 desde la izquierda y ajustar el top = 50 desde arriba
root.geometry("500x300+100+50")

#3. Bucle princiapal, mantiene la ventana viva y escucha eventos
root.mainloop()